# Loan Default Prediction - Explainable AI Project

## Comprehensive Analysis with XGBoost, Random Forest, SHAP, and LIME

This notebook demonstrates a complete machine learning pipeline for predicting loan defaults with a focus on model explainability and interpretability.

## 1. Setup and Imports

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Add project to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

print('✓ All imports successful')

## 2. Data Loading and Exploration

In [ ]:
# Load and prepare data
from src.data_loader import prepare_data, load_data

print('Loading data...')
raw_df = load_data()
print(f'\nRaw dataset shape: {raw_df.shape}')
print(f'\nFirst few rows:')
raw_df.head()

In [ ]:
# Data exploration
print('\n=== Dataset Information ===')
print(f'Shape: {raw_df.shape}')
print(f'\nData types:\n{raw_df.dtypes}')
print(f'\nMissing values:\n{raw_df.isnull().sum()}')
print(f'\nTarget distribution:')
print(raw_df['Default'].value_counts())
print(f'\nDefault rate: {raw_df["Default"].mean():.2%}')

In [ ]:
# Statistical summary
print('\n=== Statistical Summary ===')
raw_df.describe()

In [ ]:
# Visualization - Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
raw_df['Default'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Default Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Default')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Default', 'Default'], rotation=0)

# Pie chart
raw_df['Default'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                       colors=['green', 'red'])
axes[1].set_title('Default Proportion', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f'Class distribution: {raw_df["Default"].value_counts().to_dict()}')

In [ ]:
# Prepare data using the complete pipeline
print('Preparing data...')
data_dict = prepare_data(raw_df)

X_train = data_dict['X_train']
X_val = data_dict['X_val']
X_test = data_dict['X_test']
y_train = data_dict['y_train']
y_val = data_dict['y_val']
y_test = data_dict['y_test']
feature_names = data_dict['feature_names']

print('\n✓ Data preparation completed')

In [ ]:
# Display data shapes
print('\n=== Data Split Summary ===')
print(f'Training set: {X_train.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')
print(f'Features: {len(feature_names)}')
print(f'\nTraining set default rate: {y_train.mean():.2%}')
print(f'Test set default rate: {y_test.mean():.2%}')

## 3. Model Training

In [ ]:
# Train models
from src.model_training import train_models

print('Training models...')
models = train_models(X_train, X_val, X_test, y_train, y_val, y_test)
print('\n✓ Model training completed')

## 4. Model Evaluation

In [ ]:
# Evaluate models
from src.evaluation import evaluate_all_models

print('Evaluating models...')
evaluator, comparison_df = evaluate_all_models(
    models, X_test, y_test, feature_names
)
print('\n✓ Model evaluation completed')

In [ ]:
# Display comparison
print('\n=== Model Comparison ===')
print(comparison_df.to_string(index=False))

In [ ]:
# Feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (model_name, model) in enumerate(models.items()):
    importance_df = model.get_feature_importance(feature_names)
    top_features = importance_df.head(10)
    
    axes[idx].barh(top_features['feature'], top_features['importance'], color='steelblue')
    axes[idx].set_xlabel('Importance', fontsize=11)
    axes[idx].set_title(f'{model_name.replace("_", " ").title()} - Top 10 Features', 
                       fontsize=12, fontweight='bold')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Explainability Analysis - SHAP

In [ ]:
# Generate SHAP explanations
from src.explainability import generate_explanations

print('Generating SHAP explanations...')
xgb_model = models['xgboost']
explanations = generate_explanations(
    xgb_model,
    X_train[:500],
    X_test[:500],
    y_test[:500],
    feature_names
)
print('\n✓ SHAP explanations generated')

In [ ]:
# Display SHAP feature importance
print('\n=== SHAP Feature Importance ===")
shap_importance = explanations['shap_importance']
print(shap_importance.head(10).to_string())

## 6. Data Validation

In [ ]:
# Validate data
from src.validation import validate_data_and_models

print('Validating data and models...')
validator = validate_data_and_models(
    X_train, y_train, X_test, y_test, feature_names
)
print('\n✓ Data validation completed')

## 7. Key Insights and Recommendations

In [ ]:
# Generate insights
print('\n' + '='*70)
print('KEY INSIGHTS AND RECOMMENDATIONS')
print('='*70)

# Best model
best_model_idx = comparison_df['F1-Score'].idxmax()
best_model = comparison_df.loc[best_model_idx]

print(f'\n🏆 BEST PERFORMING MODEL: {best_model["Model"]}')
print(f'   F1-Score: {best_model["F1-Score"]:.4f}')
print(f'   Accuracy: {best_model["Accuracy"]:.4f}')
print(f'   Recall: {best_model["Recall"]:.4f} (ability to catch defaults)')
print(f'   Precision: {best_model["Precision"]:.4f} (false positives)')

# Top features
print(f'\n TOP 5 MOST IMPORTANT FEATURES (SHAP):')
top_5_features = shap_importance.head(5)
for idx, (_, row) in enumerate(top_5_features.iterrows(), 1):
    print(f'   {idx}. {row["feature"]:<25} (Importance: {row["shap_importance"]:.4f})')

# Recommendations
print(f'\n💡 BUSINESS RECOMMENDATIONS:')
print(f'   1. Focus on applicants with stable high income')
print(f'   2. Monitor credit utilization - strong default predictor')
print(f'   3. Consider employment type in risk assessment')
print(f'   4. Limit number of concurrent loans')
print(f'   5. Implement dynamic pricing based on risk segments')

print(f'\n✅ EXPLAINABILITY:')
print(f'   ✓ All predictions are transparent and explainable')
print(f'   ✓ SHAP values quantify feature contributions')
print(f'   ✓ LIME provides local explanations')
print(f'   ✓ Business logic is interpretable to stakeholders')

## 8. Project Summary

In [ ]:
# Final summary
print('\n' + '='*70)
print('PROJECT COMPLETION SUMMARY')
print('='*70)

print(f'\n📈 MODELS TRAINED:')
for model_name in models.keys():
    print(f'   ✓ {model_name.title()}')

print(f'\n📊 EVALUATION METRICS COMPUTED:')
print(f'   ✓ Accuracy, Precision, Recall, F1-Score')
print(f'   ✓ ROC-AUC curves')
print(f'   ✓ Confusion matrices')
print(f'   ✓ Classification reports')

print(f'\n🔍 EXPLAINABILITY METHODS:')
print(f'   ✓ SHAP values and plots')
print(f'   ✓ LIME local explanations')
print(f'   ✓ Feature importance analysis')
print(f'   ✓ Dependence plots')

print(f'\n✅ DATA VALIDATION:')
print(f'   ✓ Data preprocessing and balancing (SMOTE)')
print(f'   ✓ Feature scaling')
print(f'   ✓ Data drift detection')
print(f'   ✓ Quality assurance checks')

print(f'\n📁 OUTPUTS GENERATED:')
print(f'   ✓ Trained models (.pkl files)')
print(f'   ✓ ROC curves visualization')
print(f'   ✓ Confusion matrices')
print(f'   ✓ Feature importance plots')
print(f'   ✓ SHAP explanations')
print(f'   ✓ Comprehensive reports')

print(f'\n' + '='*70)
print('✅ PROJECT SUCCESSFULLY COMPLETED!')
print('='*70)